# Module 11 Exercise (Solution): Hungarian matching from scratch + applied pretrained DETR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/11-detr/exercise_solution.ipynb)

Module page: [Module 11: DETR — Object Detection with Transformers](https://nsteve2407.github.io/llm-transformers-course/modules/11-detr/)

Classical detectors (Faster R-CNN, YOLO) predict *far more* boxes than there are objects -- hundreds to
thousands of anchors or region proposals -- and then rely on post-hoc heuristics (non-max suppression, NMS)
to throw away near-duplicates. DETR reframes detection as **direct set prediction**: a fixed number of
*object queries* attend over the image and each independently predicts one box (or "no object"). Training
needs a way to decide, for each ground-truth object, *which* query is responsible for it -- **bipartite
matching** via the **Hungarian algorithm**, using a cost that combines classification confidence, L1 box
distance, and **generalized IoU (GIoU)**. This notebook has two parts:

1. **Part A -- Hungarian matching from scratch**: a synthetic toy scene (3 ground-truth objects, 5 predicted
   queries) with a **known correct** assignment by construction. `box_iou`/`generalized_box_iou` implemented
   from scratch and unit-tested against hand-computed values. The 5x3 matching cost matrix (classification +
   L1 + GIoU). `scipy.optimize.linear_sum_assignment` solves it, and the result is checked against the
   known-correct assignment. The full Hungarian/set-prediction loss (matched-pair NLL + box L1/GIoU,
   down-weighted no-object loss). A concrete permutation-invariance check: shuffling the ground-truth order
   changes the raw index labels but not the total matched cost or the semantic query-to-object pairing.
2. **Part B -- applied pretrained DETR**: load `facebook/detr-resnet-50`, run inference on real photos,
   visualize predicted boxes, extract and visualize **object-query cross-attention** (which pixels a query
   that matched a real object actually attends to), and fine-tune on a small custom detection set with a
   pre-/post-fine-tuning comparison via `torchmetrics`'s mAP.

In [ ]:
try:
    import transformers
except ImportError:
    %pip install -q transformers

In [ ]:
# timm provides the CNN backbone (ResNet-50) that DetrForObjectDetection.from_pretrained loads weights into.
try:
    import timm
except ImportError:
    %pip install -q timm

In [ ]:
# torchmetrics supplies the mAP (mean Average Precision) metric used for the Part B pre-/post-fine-tuning
# comparison. (scipy.optimize.linear_sum_assignment, used throughout Part A, ships with scipy -- already a
# transitive dependency here, so it needs no guarded install of its own.)
try:
    import torchmetrics
except ImportError:
    %pip install -q torchmetrics

In [ ]:
%matplotlib inline
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from transformers import DetrFeatureExtractor, DetrForObjectDetection

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
HAS_CUDA = torch.cuda.is_available()
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}, HAS_CUDA={HAS_CUDA}")

## Design choices and judgment calls (documented up front)

- **`SMOKE_TEST` scope**: shrinks only the Part B *fine-tuning* subset size and epoch count. Part A's
  from-scratch math (unit tests, matching, permutation-invariance check) always runs at full rigor -- it's
  cheap. Part B's pretrained-DETR inference and cross-attention visualization always run on the same 3 real
  photos with the same real `facebook/detr-resnet-50` weights, whether or not `SMOKE_TEST` is set -- the
  whole point of Part B is exercising the real pretrained model, so nothing about *that* is allowed to
  shrink.
- **`transformers==4.18.0`'s `DetrFeatureExtractor.size` bug**: `facebook/detr-resnet-50`'s hosted
  `preprocessor_config.json` stores `"size": {"shortest_edge": 800, "longest_edge": 1333}` (the newer,
  dict-based convention used by `DetrImageProcessor` in later `transformers` versions). This version's
  `DetrFeatureExtractor._resize` expects `self.size` to be a **plain int** and `self.max_size` separately --
  passing the dict straight through raises `TypeError: unsupported operand type(s) for *: 'float' and
  'dict'` the moment an image is preprocessed. The fix used below is to pass `size=800, max_size=1333`
  explicitly to `from_pretrained(...)`, which overrides the (broken) checkpoint values with the equivalent
  int/int pair DETR was actually trained with -- same effective preprocessing, just expressed in the form
  this `transformers` version's code path can handle. This is a version-compatibility workaround, not a
  change to DETR's preprocessing semantics.
- **Matching-cost / loss weights**: `LAMBDA_CLS=1`, `LAMBDA_L1=5`, `LAMBDA_GIOU=2` throughout, matching the
  DETR paper's Hungarian-matching cost and set-prediction loss weights exactly (the same three weights are
  reused for both the matching cost *and* the training loss, as in the paper).
- **No-object down-weighting (`eos_coef`)**: with 100 queries but typically only a handful of real objects
  per image, most queries are correctly assigned "no object" (∅). Weighting the ∅ class equally with real
  classes in the classification loss would let a trivial "always predict ∅" solution dominate the gradient.
  `eos_coef=0.1` (the paper's value) down-weights the no-object term relative to real-object terms.
- **Image source for Part B inference (B2/B3)**: 3 real COCO `val2017` photos, fetched directly from
  `images.cocodataset.org` (the same URLs used in HF's own DETR usage examples) -- real, natural photographs
  DETR was actually pretrained to detect objects in, unlike CIFAR-10's 32x32 thumbnails. This is a stronger
  and more honest demonstration of the pretrained model than synthetic or heavily-downscaled images would be.
- **Fine-tuning dataset (B4)**: a small **synthetic** shape-detection task (colored circles/squares/triangles
  drawn with `PIL.ImageDraw` at random positions on a fixed-size canvas) rather than a COCO subset. This
  keeps the exercise fully self-contained (no COCO annotation file to download/parse) while still exercising
  the *real* fine-tuning path: `DetrForObjectDetection.from_pretrained(..., num_labels=3,
  ignore_mismatched_sizes=True)` reinitializes only the classification head (the box-regression head keeps
  its pretrained weights), and `model(pixel_values=..., labels=...)` runs transformers' own built-in
  Hungarian matcher + set-prediction loss (`DetrHungarianMatcher` / `DetrLoss`) -- the exact same mechanism
  built from scratch in Part A, now driving real gradient updates on a real (if small) pretrained model.

## Part A: Hungarian matching from scratch

### A1. Synthetic toy scene: a known-correct assignment by construction

3 ground-truth objects (a cat, a dog, a car) and 5 predicted object queries, in normalized `(cx, cy, w, h)`
format (values in `[0, 1]`, independent of image size -- exactly the format DETR's box head regresses).
Queries 0/1/2 are deliberately built to sit almost exactly on top of GT 0/1/2 respectively, with class
logits strongly favoring the *correct* class. Queries 3/4 are placed far from every GT box, with logits
strongly favoring the no-object (∅) class. By construction, the only sensible optimal assignment is
`query0<->GT0, query1<->GT1, query2<->GT2`, with queries 3 and 4 left unmatched ("no object") -- this lets
A5 below *check* the Hungarian solver's output against a known ground truth, not just eyeball plausibility.

In [ ]:
CLASS_NAMES = ["cat", "dog", "car"]
NUM_CLASSES = len(CLASS_NAMES)
NO_OBJECT = NUM_CLASSES  # class index NUM_CLASSES is reserved for "no object" (matches DETR's convention)

# Ground truth: 3 objects, normalized (cx, cy, w, h).
gt_boxes = torch.tensor([
    [0.20, 0.20, 0.20, 0.20],  # cat, upper-left
    [0.50, 0.50, 0.30, 0.30],  # dog, center
    [0.80, 0.30, 0.15, 0.25],  # car, upper-right
])
gt_labels = torch.tensor([0, 1, 2])  # cat, dog, car

# 5 predicted queries: boxes (normalized cx, cy, w, h) and raw class logits over
# [cat, dog, car, no-object]. Queries 0/1/2 are near-exact matches (by construction) for GT 0/1/2 with
# confident correct-class logits; queries 3/4 are far from every GT box with confident no-object logits.
pred_boxes = torch.tensor([
    [0.21, 0.19, 0.22, 0.18],  # near GT0 (cat)
    [0.52, 0.48, 0.31, 0.29],  # near GT1 (dog)
    [0.78, 0.32, 0.16, 0.24],  # near GT2 (car)
    [0.05, 0.90, 0.10, 0.10],  # far from every GT -> should match "no object"
    [0.95, 0.05, 0.08, 0.08],  # far from every GT -> should match "no object"
])
pred_logits = torch.tensor([
    [4.0, -1.0, -1.0, 0.0],   # confident: cat
    [-1.0, 4.0, -1.0, 0.0],   # confident: dog
    [-1.0, -1.0, 4.0, 0.0],   # confident: car
    [-1.0, -1.0, -1.0, 4.0],  # confident: no-object
    [-1.0, -1.0, -1.0, 4.0],  # confident: no-object
])
NUM_QUERIES = pred_boxes.shape[0]

KNOWN_CORRECT_QUERY_TO_GT = {0: 0, 1: 1, 2: 2}  # query idx -> GT idx
KNOWN_CORRECT_UNMATCHED = {3, 4}                # query idx -> "no object"
print(f"{gt_boxes.shape[0]} ground-truth objects, {NUM_QUERIES} predicted queries")

In [ ]:
# Visualize the toy scene: solid boxes = ground truth, dashed boxes = predicted queries, in the normalized
# unit square. Purely illustrative -- makes the "by-construction" claim above visually checkable.
fig, ax = plt.subplots(figsize=(5, 5))
gt_colors = ["tab:red", "tab:green", "tab:blue"]
for i, (box, label) in enumerate(zip(gt_boxes, gt_labels)):
    cx, cy, w, h = box.tolist()
    ax.add_patch(plt.Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False,
                                edgecolor=gt_colors[label], linewidth=2.5))
    ax.text(cx - w / 2, cy - h / 2 - 0.02, f"GT{i}:{CLASS_NAMES[label]}", color=gt_colors[label], fontsize=9)
for i, box in enumerate(pred_boxes):
    cx, cy, w, h = box.tolist()
    pred_cls = pred_logits[i, :NUM_CLASSES].argmax().item()
    color = gt_colors[pred_cls] if pred_logits[i].argmax().item() != NO_OBJECT else "gray"
    ax.add_patch(plt.Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False,
                                edgecolor=color, linewidth=1.5, linestyle="--"))
    ax.text(cx - w / 2, cy + h / 2 + 0.01, f"q{i}", color=color, fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_aspect("equal")
ax.set_title("Toy scene: solid = ground truth, dashed = predicted queries")
plt.show()

### A2. `box_iou` and `generalized_box_iou` from scratch

Both take boxes in `(x1, y1, x2, y2)` format, so `(cx, cy, w, h)` predictions/targets are converted first.
GIoU = IoU − (area of the smallest enclosing box **not** covered by either box) / (area of the enclosing
box) = IoU − (`C − union) / C`, where `C` is the area of the smallest axis-aligned box containing both
inputs. Unlike plain IoU, GIoU is well-defined (and informative -- not just stuck at 0) for boxes that don't
overlap at all, which is exactly why DETR's matching cost uses GIoU instead of IoU: for two non-overlapping
boxes, IoU's gradient with respect to box coordinates is *zero everywhere* (moving a non-overlapping box
around doesn't change "0% overlap" until it starts to touch), so IoU alone gives the matcher/optimizer no
signal about *how far off* an unmatched prediction is. GIoU keeps decreasing (more negative) the farther
apart two boxes are, since a larger enclosing box means more of `C` is "wasted" on neither box.

In [ ]:
def box_cxcywh_to_xyxy(boxes):
    """boxes: (..., 4) as (cx, cy, w, h) -> (..., 4) as (x1, y1, x2, y2)."""
    cx, cy, w, h = boxes.unbind(-1)
    x1 = cx - 0.5 * w
    y1 = cy - 0.5 * h
    x2 = cx + 0.5 * w
    y2 = cy + 0.5 * h
    return torch.stack([x1, y1, x2, y2], dim=-1)


def box_iou(boxes1, boxes2):
    """boxes1: (N, 4), boxes2: (M, 4), both (x1, y1, x2, y2). Returns (iou, union), each (N, M)."""
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])

    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])  # (N, M, 2) intersection top-left
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])  # (N, M, 2) intersection bottom-right
    wh = (rb - lt).clamp(min=0)                               # 0 if boxes don't overlap on that axis
    inter = wh[:, :, 0] * wh[:, :, 1]

    union = area1[:, None] + area2[None, :] - inter
    iou = inter / union
    return iou, union


def generalized_box_iou(boxes1, boxes2):
    """boxes1: (N, 4), boxes2: (M, 4), both (x1, y1, x2, y2). Returns giou, (N, M), each entry in [-1, 1]."""
    iou, union = box_iou(boxes1, boxes2)

    lt = torch.min(boxes1[:, None, :2], boxes2[None, :, :2])  # smallest enclosing box top-left
    rb = torch.max(boxes1[:, None, 2:], boxes2[None, :, 2:])  # smallest enclosing box bottom-right
    wh = (rb - lt).clamp(min=0)
    area_c = wh[:, :, 0] * wh[:, :, 1]

    giou = iou - (area_c - union) / area_c
    return giou

### A3. Unit tests: identical / disjoint / hand-computed partial overlap

- **(a) Identical boxes**: IoU = GIoU = 1.0 exactly (full overlap, and the enclosing box exactly equals the
  union, so the GIoU penalty term is 0).
- **(b) Disjoint, far apart**: IoU = 0 (no overlap at all). GIoU must be **strictly negative** -- with the
  boxes far apart, the enclosing box is much larger than their (disjoint) union, so a large fraction of `C`
  is "wasted", pushing GIoU well below 0 (not just barely negative).
- **(c) Hand-computed partial overlap**: box1 = `(cx=1, cy=1, w=2, h=2)` -> xyxy `(0, 0, 2, 2)`; box2 =
  `(cx=2, cy=2, w=2, h=2)` -> xyxy `(1, 1, 3, 3)`. By hand:
  - Intersection: `x in [1, 2]`, `y in [1, 2]` -> a 1x1 square, area = **1**.
  - Areas: box1 = box2 = 2x2 = 4 each. Union = 4 + 4 − 1 = **7**. So **IoU = 1/7 ≈ 0.142857**.
  - Smallest enclosing box: `(0, 0, 3, 3)`, area = **9**. GIoU = IoU − (C − union)/C = 1/7 − (9 − 7)/9
    = 1/7 − 2/9 = **9/63 − 14/63 = −5/63 ≈ −0.079365**.

In [ ]:
# (a) Identical boxes -> IoU = GIoU = 1.0
identical_boxes = torch.tensor([[0.1, 0.1, 0.4, 0.4], [0.5, 0.5, 0.9, 0.9]])
iou_a, _ = box_iou(identical_boxes, identical_boxes)
giou_a = generalized_box_iou(identical_boxes, identical_boxes)
assert torch.allclose(iou_a.diag(), torch.ones(2)), f"expected IoU=1.0 on the diagonal, got {iou_a.diag()}"
assert torch.allclose(giou_a.diag(), torch.ones(2)), f"expected GIoU=1.0 on the diagonal, got {giou_a.diag()}"
print("(a) identical boxes: IoU = GIoU = 1.0 -- OK")

# (b) Disjoint, far apart -> IoU = 0, GIoU strictly negative
far1 = torch.tensor([[0.0, 0.0, 0.1, 0.1]])
far2 = torch.tensor([[0.8, 0.8, 0.9, 0.9]])
iou_b, _ = box_iou(far1, far2)
giou_b = generalized_box_iou(far1, far2)
assert torch.allclose(iou_b, torch.zeros(1, 1)), f"expected IoU=0, got {iou_b}"
assert (giou_b < 0).all(), f"expected GIoU strictly negative, got {giou_b}"
print(f"(b) disjoint far-apart boxes: IoU = {iou_b.item():.4f}, GIoU = {giou_b.item():.4f} (< 0) -- OK")

# (c) Hand-computed partial overlap (see markdown above)
box1_cxcywh = torch.tensor([[1.0, 1.0, 2.0, 2.0]])
box2_cxcywh = torch.tensor([[2.0, 2.0, 2.0, 2.0]])
box1_xyxy = box_cxcywh_to_xyxy(box1_cxcywh)
box2_xyxy = box_cxcywh_to_xyxy(box2_cxcywh)
assert torch.allclose(box1_xyxy, torch.tensor([[0.0, 0.0, 2.0, 2.0]]))
assert torch.allclose(box2_xyxy, torch.tensor([[1.0, 1.0, 3.0, 3.0]]))

iou_c, _ = box_iou(box1_xyxy, box2_xyxy)
giou_c = generalized_box_iou(box1_xyxy, box2_xyxy)
expected_iou = 1.0 / 7.0
expected_giou = 1.0 / 7.0 - 2.0 / 9.0
assert torch.allclose(iou_c, torch.tensor([[expected_iou]]), atol=1e-6), (iou_c, expected_iou)
assert torch.allclose(giou_c, torch.tensor([[expected_giou]]), atol=1e-6), (giou_c, expected_giou)
print(f"(c) hand-computed partial overlap: IoU = {iou_c.item():.6f} (expected {expected_iou:.6f}), "
      f"GIoU = {giou_c.item():.6f} (expected {expected_giou:.6f}) -- OK")

print("All GIoU unit tests passed.")

### A4. Matching cost matrix (classification + L1 + GIoU)

For every (query, GT) pair, combine three costs (lower = better match), matching the DETR paper's weights:

- **Classification cost**: `-p_i[gt_class_j]`, the *negative* predicted probability the query assigns to
  the GT's class (so a confident correct prediction has a very negative -- low -- cost).
- **Box L1 cost**: `||pred_box_i - gt_box_j||_1` in normalized `(cx, cy, w, h)` space.
- **GIoU cost**: `-generalized_box_iou(pred_box_i, gt_box_j)` (negated, so higher overlap/GIoU means lower
  cost, consistent with "lower cost = better match").

`cost = LAMBDA_CLS * cost_class + LAMBDA_L1 * cost_bbox + LAMBDA_GIOU * cost_giou`.

In [ ]:
LAMBDA_CLS = 1.0
LAMBDA_L1 = 5.0
LAMBDA_GIOU = 2.0


def build_cost_matrix(pred_boxes, pred_logits, gt_boxes, gt_labels):
    """pred_boxes: (Q, 4) cxcywh. pred_logits: (Q, num_classes + 1) raw logits (last column = no-object).
    gt_boxes: (G, 4) cxcywh. gt_labels: (G,) int64 class indices. Returns the (Q, G) matching cost matrix."""
    probs = pred_logits.softmax(-1)              # (Q, num_classes + 1)
    cost_class = -probs[:, gt_labels]             # (Q, G): negative prob of each GT's class, per query

    cost_bbox = torch.cdist(pred_boxes, gt_boxes, p=1)  # (Q, G) L1 distance in cxcywh space

    pred_xyxy = box_cxcywh_to_xyxy(pred_boxes)
    gt_xyxy = box_cxcywh_to_xyxy(gt_boxes)
    cost_giou = -generalized_box_iou(pred_xyxy, gt_xyxy)  # (Q, G)

    return LAMBDA_CLS * cost_class + LAMBDA_L1 * cost_bbox + LAMBDA_GIOU * cost_giou

### A5. Solve with `scipy.optimize.linear_sum_assignment`

`linear_sum_assignment` solves the Hungarian algorithm's assignment problem exactly: given the `(Q, G)` cost
matrix, it finds the one-to-one matching between (a subset of) queries and GT boxes minimizing total cost.
With `Q=5 > G=3`, exactly 3 queries get matched (one per GT box) and the other 2 are left unmatched -- those
2 are implicitly "no object". This is checked against the known-correct assignment built into the toy scene
in A1.

In [ ]:
cost_matrix = build_cost_matrix(pred_boxes, pred_logits, gt_boxes, gt_labels)
print("Cost matrix (rows = queries, columns = GT boxes):")
print(cost_matrix)

row_ind, col_ind = linear_sum_assignment(cost_matrix.numpy())
print(f"\nOptimal assignment: queries {row_ind.tolist()} <-> GT {col_ind.tolist()}")

matched_query_to_gt = dict(zip(row_ind.tolist(), col_ind.tolist()))
unmatched_queries = set(range(NUM_QUERIES)) - set(row_ind.tolist())
for q in range(NUM_QUERIES):
    if q in matched_query_to_gt:
        g = matched_query_to_gt[q]
        print(f"  query {q} <-> GT {g} ({CLASS_NAMES[gt_labels[g]]})")
    else:
        print(f"  query {q} <-> no object")

# Check against the by-construction known-correct assignment from A1.
assert matched_query_to_gt == KNOWN_CORRECT_QUERY_TO_GT, (matched_query_to_gt, KNOWN_CORRECT_QUERY_TO_GT)
assert unmatched_queries == KNOWN_CORRECT_UNMATCHED, (unmatched_queries, KNOWN_CORRECT_UNMATCHED)
print("\nMatches the known-correct assignment from A1 -- OK")

### A6. Full Hungarian / set-prediction loss

Given the optimal assignment, the total DETR set-prediction loss for this image is:

- **Classification loss**: a single weighted cross-entropy over *all* `Q` queries at once. The target for
  each matched query is its GT's class; the target for every unmatched query is the no-object class. The
  no-object class's weight is down-weighted by `eos_coef` (see design notes above) -- this is exactly what
  "the down-weighted no-object loss for unmatched queries" means in practice: one cross-entropy call with a
  non-uniform class-weight vector, not a separate loss term.
- **Box L1 loss** and **GIoU loss**: computed *only* over the matched pairs (unmatched queries have no GT
  box to regress toward), normalized by the number of matched pairs.
- **Total** = `LAMBDA_CLS * loss_ce + LAMBDA_L1 * loss_bbox + LAMBDA_GIOU * loss_giou`.

In [ ]:
def compute_hungarian_loss(pred_boxes, pred_logits, gt_boxes, gt_labels, row_ind, col_ind,
                            num_classes, eos_coef=0.1):
    """pred_boxes: (Q, 4) cxcywh. pred_logits: (Q, num_classes + 1). gt_boxes: (G, 4) cxcywh.
    gt_labels: (G,) int64. row_ind/col_ind: matched (query_idx, gt_idx) arrays from linear_sum_assignment.
    Returns (total_loss, {"loss_ce": ..., "loss_bbox": ..., "loss_giou": ...})."""
    num_queries = pred_boxes.shape[0]
    no_object = num_classes
    matched_q = torch.as_tensor(row_ind, dtype=torch.long)
    matched_g = torch.as_tensor(col_ind, dtype=torch.long)

    # Classification: every query defaults to "no object"; matched queries get their GT's class.
    target_classes = torch.full((num_queries,), no_object, dtype=torch.long)
    target_classes[matched_q] = gt_labels[matched_g]
    class_weight = torch.ones(num_classes + 1)
    class_weight[no_object] = eos_coef
    loss_ce = F.cross_entropy(pred_logits, target_classes, weight=class_weight)

    # Box losses: only over matched pairs.
    matched_pred_boxes = pred_boxes[matched_q]
    matched_gt_boxes = gt_boxes[matched_g]
    num_matched = matched_q.numel()
    loss_bbox = F.l1_loss(matched_pred_boxes, matched_gt_boxes, reduction="sum") / num_matched

    matched_pred_xyxy = box_cxcywh_to_xyxy(matched_pred_boxes)
    matched_gt_xyxy = box_cxcywh_to_xyxy(matched_gt_boxes)
    giou_matched = torch.diag(generalized_box_iou(matched_pred_xyxy, matched_gt_xyxy))
    loss_giou = (1.0 - giou_matched).sum() / num_matched

    total = LAMBDA_CLS * loss_ce + LAMBDA_L1 * loss_bbox + LAMBDA_GIOU * loss_giou
    return total, {"loss_ce": loss_ce, "loss_bbox": loss_bbox, "loss_giou": loss_giou}


total_loss, loss_components = compute_hungarian_loss(
    pred_boxes, pred_logits, gt_boxes, gt_labels, row_ind, col_ind, NUM_CLASSES
)
for name, value in loss_components.items():
    print(f"{name}: {value.item():.4f}")
print(f"total Hungarian/set-prediction loss: {total_loss.item():.4f}")

# Sanity checks: all loss terms must be non-negative (cross-entropy, L1, and 1-GIoU each are by
# construction), and this well-matched toy scene should produce a *small* loss.
assert loss_components["loss_ce"].item() >= 0
assert loss_components["loss_bbox"].item() >= 0
assert loss_components["loss_giou"].item() >= 0
assert total_loss.item() < 2.0, f"expected a small loss for a near-perfect toy assignment, got {total_loss.item()}"
print("Loss sanity checks passed (all components non-negative, total loss small for a near-perfect match).")

### A7. Permutation invariance (concrete verification)

Bipartite matching should not care what *order* the ground-truth boxes are listed in -- shuffling them just
relabels the columns of the cost matrix; the actual optimal pairing (which query goes with which physical
object) must be unchanged. Two things are checked concretely (not just asserted by inspection):

- **(a) Cost equality**: the total assigned cost from `linear_sum_assignment` on the shuffled cost matrix
  must equal the original total cost (within float tolerance) -- the *value* of the optimum can't depend on
  column order.
- **(b) Semantic pairing equality**: comparing raw indices after shuffling is meaningless (GT index `1` in
  the shuffled order is a *different physical object* than GT index `1` originally). Instead, each matched
  query's paired GT is compared by mapping the shuffled column index back to the *original* GT identity
  (via the permutation) -- and separately, by directly comparing the paired GT box's *coordinates and class
  label*, not just its index. Both must agree with the original (unshuffled) pairing.

In [ ]:
perm = torch.tensor([2, 0, 1])  # an arbitrary reordering of the 3 GT boxes
gt_boxes_shuffled = gt_boxes[perm]
gt_labels_shuffled = gt_labels[perm]

cost_matrix_shuffled = build_cost_matrix(pred_boxes, pred_logits, gt_boxes_shuffled, gt_labels_shuffled)
row_ind_shuf, col_ind_shuf = linear_sum_assignment(cost_matrix_shuffled.numpy())

total_cost_orig = cost_matrix[row_ind, col_ind].sum()
total_cost_shuf = cost_matrix_shuffled[row_ind_shuf, col_ind_shuf].sum()
print(f"total cost (original GT order):  {total_cost_orig.item():.6f}")
print(f"total cost (shuffled GT order):  {total_cost_shuf.item():.6f}")
assert torch.allclose(total_cost_orig, total_cost_shuf, atol=1e-5), (total_cost_orig, total_cost_shuf)
print("(a) total assigned cost is numerically identical -- OK")

# (b) Semantic pairing: map the shuffled run's column indices back to ORIGINAL GT indices via perm, and
# separately compare by box coordinates + class label directly -- raw indices alone would be misleading
# here since col_ind_shuf indexes into the shuffled array, not the original one.
orig_pairing_by_gt_idx = {int(q): int(g) for q, g in zip(row_ind, col_ind)}
shuf_pairing_mapped_to_orig_idx = {int(q): int(perm[g]) for q, g in zip(row_ind_shuf, col_ind_shuf)}
print(f"query -> original GT idx (unshuffled run): {orig_pairing_by_gt_idx}")
print(f"query -> original GT idx (shuffled run, mapped back via perm): {shuf_pairing_mapped_to_orig_idx}")
assert orig_pairing_by_gt_idx == shuf_pairing_mapped_to_orig_idx

orig_pairing_by_box = {int(q): (tuple(gt_boxes[g].tolist()), int(gt_labels[g])) for q, g in zip(row_ind, col_ind)}
shuf_pairing_by_box = {int(q): (tuple(gt_boxes_shuffled[g].tolist()), int(gt_labels_shuffled[g]))
                        for q, g in zip(row_ind_shuf, col_ind_shuf)}
assert orig_pairing_by_box == shuf_pairing_by_box, (orig_pairing_by_box, shuf_pairing_by_box)
print("(b) semantic query-to-object pairing (by box coordinates + class, not raw index) is unchanged -- OK")

print("\nPermutation invariance verified: shuffling GT order changes column labels but not the optimum's "
      "value or the physical query-to-object pairing it represents.")

### Part A summary

`generalized_box_iou` matches the standard GIoU formula on 3 independently-verified cases (including a fully
hand-computed partial-overlap case), the matching cost matrix combines classification + L1 + GIoU cost
exactly as in the DETR paper, `scipy.optimize.linear_sum_assignment` recovers the known-correct assignment
on a toy scene built so the right answer is unambiguous, the full set-prediction loss combines matched-pair
classification/box losses with a down-weighted no-object term, and the whole pipeline has been shown --
concretely, not just asserted -- to be invariant to the arbitrary order ground-truth objects happen to be
listed in. Part B below applies the *real* version of this exact mechanism (transformers' built-in
`DetrHungarianMatcher`/`DetrLoss`) inside a real pretrained model.

## Part B: pretrained DETR -- inference, attention, fine-tuning

### B1. Load `facebook/detr-resnet-50`

`size=800, max_size=1333` is passed explicitly to the feature extractor to work around the
`transformers==4.18.0` size-dict bug described in the design notes above (same effective preprocessing DETR
was trained with).

In [ ]:
DETR_CHECKPOINT = "facebook/detr-resnet-50"
feature_extractor = DetrFeatureExtractor.from_pretrained(DETR_CHECKPOINT, size=800, max_size=1333)
detr_model = DetrForObjectDetection.from_pretrained(DETR_CHECKPOINT)
detr_model.eval().to(device)

n_params = sum(p.numel() for p in detr_model.parameters())
print(f"Loaded {detr_model.name_or_path} ({n_params / 1e6:.1f}M params) onto {device}")
print(f"COCO classes: {len(detr_model.config.id2label)} (incl. unused 'N/A' placeholders)")
assert n_params > 4e7, f"expected ~41.5M params for a real detr-resnet-50 checkpoint, got {n_params}"

### B2. Inference on real images + box visualization

3 real photos from the COCO `val2017` split, fetched directly from `images.cocodataset.org` (see design
notes above for why real photos rather than CIFAR-10 thumbnails). Boxes with predicted-class confidence
above 0.7 are kept and drawn with their class label and score.

In [ ]:
SAMPLE_IMAGE_URLS = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",  # two cats on a couch, with remotes
    "http://images.cocodataset.org/val2017/000000037777.jpg",  # kitchen counter with fruit/oranges
    "http://images.cocodataset.org/val2017/000000252219.jpg",  # people with umbrellas
]
CONFIDENCE_THRESHOLD = 0.7


def load_image_from_url(url):
    response = requests.get(url, stream=True, timeout=30)
    response.raise_for_status()
    return Image.open(response.raw).convert("RGB")


def run_detr_inference(image, model, feature_extractor, threshold=CONFIDENCE_THRESHOLD):
    """Runs DETR on a single PIL image. Returns kept boxes/scores/labels (absolute pixel xyxy) plus the
    raw model outputs (with attentions) and feature-extractor inputs, for reuse in B3."""
    inputs = feature_extractor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    target_sizes = torch.tensor([image.size[::-1]], device=device)  # (height, width)
    result = feature_extractor.post_process(outputs, target_sizes=target_sizes)[0]
    keep = result["scores"] > threshold
    return {
        "boxes": result["boxes"][keep].cpu(),
        "scores": result["scores"][keep].cpu(),
        "labels": result["labels"][keep].cpu(),
        "kept_query_idx": keep.nonzero().flatten().cpu(),
        "outputs": outputs,
        "inputs": inputs,
    }


sample_images = [load_image_from_url(u) for u in SAMPLE_IMAGE_URLS]
inference_results = [run_detr_inference(img, detr_model, feature_extractor) for img in sample_images]
for url, image, result in zip(SAMPLE_IMAGE_URLS, sample_images, inference_results):
    names = [detr_model.config.id2label[l.item()] for l in result["labels"]]
    print(f"{url.rsplit('/', 1)[-1]}: {image.size}, {len(names)} boxes above {CONFIDENCE_THRESHOLD} -> {names}")

# A randomly-initialized head would essentially never produce multiple high-confidence (>0.7), semantically
# correct COCO class names across 3 different photos -- this is concrete evidence the checkpoint's real
# pretrained weights (not a random-init fallback) are what's actually running here.
assert all(len(r["labels"]) > 0 for r in inference_results), "expected at least one confident detection per image"

In [ ]:
def plot_detections(image, result, id2label, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(image)
    cmap = plt.cm.get_cmap("tab10")
    for i, (box, score, label) in enumerate(zip(result["boxes"], result["scores"], result["labels"])):
        x0, y0, x1, y1 = box.tolist()
        color = cmap(int(label) % 10)
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=2))
        text = f"{id2label[int(label)]}: {score:.2f}"
        ax.text(x0, max(y0 - 4, 0), text, fontsize=9, color="white",
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5, edgecolor="none"))
    ax.set_axis_off()
    if title:
        ax.set_title(title)
    return ax


fig, axes = plt.subplots(1, len(sample_images), figsize=(6 * len(sample_images), 6))
for ax, image, result in zip(axes, sample_images, inference_results):
    plot_detections(image, result, detr_model.config.id2label, ax=ax)
plt.tight_layout()
plt.show()

### B3. Object-query cross-attention visualization

Each of DETR's 100 object queries cross-attends over the flattened encoder feature map once per decoder
layer; `output_attentions=True` exposes these as `outputs.cross_attentions`, a tuple of one
`(batch, heads, num_queries, H'*W')` tensor per decoder layer (`H'*W'` = flattened backbone feature-map
positions). The last decoder layer's cross-attention is visualized (averaged over heads) for the queries
that matched real objects in the cats photo -- this is the attention map that shows *where in the image* a
given "this is a cat" prediction actually looked.

In [ ]:
cats_image, cats_result = sample_images[0], inference_results[0]
pixel_values = cats_result["inputs"]["pixel_values"]
H, W = pixel_values.shape[-2:]
H_feat, W_feat = math.ceil(H / 32), math.ceil(W / 32)  # ResNet-50 backbone: stride-32 feature map

cross_attn_last_layer = cats_result["outputs"].cross_attentions[-1]  # (1, heads, 100, H_feat*W_feat)
print(f"pixel_values: {tuple(pixel_values.shape)}, feature map: {H_feat}x{W_feat} = {H_feat * W_feat} "
      f"positions, cross_attentions[-1]: {tuple(cross_attn_last_layer.shape)}")
assert H_feat * W_feat == cross_attn_last_layer.shape[-1], "feature-map size must match the attention span"

attn_avg_heads = cross_attn_last_layer[0].mean(0).cpu()  # (100, H_feat*W_feat), averaged over heads

n_show = min(3, len(cats_result["kept_query_idx"]))
fig, axes = plt.subplots(1, n_show + 1, figsize=(4 * (n_show + 1), 4))
plot_detections(cats_image, cats_result, detr_model.config.id2label, ax=axes[0], title="Detections")
for i in range(n_show):
    q_idx = cats_result["kept_query_idx"][i].item()
    label_name = detr_model.config.id2label[int(cats_result["labels"][i])]
    attn_map = attn_avg_heads[q_idx].reshape(H_feat, W_feat)
    axes[i + 1].imshow(cats_image)
    axes[i + 1].imshow(attn_map, cmap="viridis", alpha=0.6, extent=(0, cats_image.width, cats_image.height, 0))
    axes[i + 1].set_axis_off()
    axes[i + 1].set_title(f"query {q_idx}: {label_name} ({cats_result['scores'][i]:.2f})")
plt.tight_layout()
plt.show()

### B4. Fine-tuning arm: a small synthetic shape-detection dataset

A self-contained detection task (no external annotation file needed): images are a fixed-size canvas with
1-3 randomly placed, non-overlapping colored shapes (circle / square / triangle), each a different "class".
Ground-truth boxes are generated alongside each image, in COCO's `[x, y, w, h]` (absolute pixel, top-left
origin) convention, which `DetrFeatureExtractor`'s `annotations=` argument consumes directly and converts to
DETR's normalized `(cx, cy, w, h)` training targets.

In [ ]:
SHAPE_CLASSES = ["circle", "square", "triangle"]
SHAPE_COLORS = [(220, 50, 50), (50, 90, 220), (40, 160, 60)]
CANVAS_SIZE = 256


def draw_shape(draw, shape, x, y, s, color):
    if shape == "circle":
        draw.ellipse([x, y, x + s, y + s], fill=color)
    elif shape == "square":
        draw.rectangle([x, y, x + s, y + s], fill=color)
    else:  # triangle
        draw.polygon([(x + s / 2, y), (x, y + s), (x + s, y + s)], fill=color)


def make_synthetic_image(rng, n_shapes=None):
    """Returns (PIL.Image, boxes, labels). boxes: list of [x, y, w, h] (absolute pixels, COCO xywh).
    labels: list of class indices into SHAPE_CLASSES."""
    image = Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), (245, 245, 245))
    draw = ImageDraw.Draw(image)
    n_shapes = n_shapes or rng.randint(1, 3)
    boxes, labels, placed = [], [], []
    attempts = 0
    while len(boxes) < n_shapes and attempts < 30:
        attempts += 1
        s = rng.randint(30, 60)
        x = rng.randint(10, CANVAS_SIZE - s - 10)
        y = rng.randint(10, CANVAS_SIZE - s - 10)
        candidate = (x, y, x + s, y + s)
        if any(not (candidate[2] < px1 or candidate[0] > px2 or candidate[3] < py1 or candidate[1] > py2)
               for (px1, py1, px2, py2) in placed):
            continue  # reject boxes that overlap an already-placed shape
        cls_idx = rng.randint(0, len(SHAPE_CLASSES) - 1)
        draw_shape(draw, SHAPE_CLASSES[cls_idx], x, y, s, SHAPE_COLORS[cls_idx])
        placed.append(candidate)
        boxes.append([x, y, s, s])
        labels.append(cls_idx)
    return image, boxes, labels


class ShapesDataset(Dataset):
    """n synthetic images (fixed seed for reproducibility), preprocessed through feature_extractor's
    COCO-style annotations= pipeline so pixel_values/labels come out in exactly the format
    DetrForObjectDetection.forward(..., labels=...) expects."""

    def __init__(self, n, seed, feature_extractor):
        rng = random.Random(seed)
        self.samples = [make_synthetic_image(rng) for _ in range(n)]
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image, boxes, labels = self.samples[idx]
        annotation = {
            "image_id": idx,
            "annotations": [
                {"bbox": box, "category_id": label, "area": box[2] * box[3], "iscrowd": 0}
                for box, label in zip(boxes, labels)
            ],
        }
        encoded = self.feature_extractor(images=[image], annotations=[annotation], return_tensors="pt")
        return encoded["pixel_values"][0], encoded["labels"][0], image


def shapes_collate_fn(batch):
    # All synthetic images are the same fixed CANVAS_SIZE square, so pixel_values stack directly with no
    # padding/pixel_mask bookkeeping needed (DETR treats a missing pixel_mask as "fully valid, no padding").
    pixel_values = torch.stack([item[0] for item in batch])
    labels = [item[1] for item in batch]
    images = [item[2] for item in batch]
    return pixel_values, labels, images


# Small fine-tuning feature extractor: same normalization as the pretrained checkpoint, but a smaller fixed
# resolution (256, matching CANVAS_SIZE) since these are small synthetic shapes, not full COCO photos.
finetune_feature_extractor = DetrFeatureExtractor.from_pretrained(DETR_CHECKPOINT, size=CANVAS_SIZE, max_size=CANVAS_SIZE)

N_TRAIN = 4 if SMOKE_TEST else 24
N_EVAL = 2 if SMOKE_TEST else 8
EPOCHS_FINETUNE = 1 if SMOKE_TEST else 8
BATCH_SIZE = 2 if SMOKE_TEST else 4
FINETUNE_LR = 1e-4

train_dataset = ShapesDataset(N_TRAIN, seed=0, feature_extractor=finetune_feature_extractor)
eval_dataset = ShapesDataset(N_EVAL, seed=1000, feature_extractor=finetune_feature_extractor)  # disjoint seed
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=shapes_collate_fn)
print(f"synthetic shapes dataset: {N_TRAIN} train / {N_EVAL} eval images, {CANVAS_SIZE}x{CANVAS_SIZE}, "
      f"{EPOCHS_FINETUNE} epoch(s), batch_size={BATCH_SIZE}")

fig, axes = plt.subplots(1, min(4, N_TRAIN), figsize=(3 * min(4, N_TRAIN), 3))
axes = [axes] if min(4, N_TRAIN) == 1 else axes
for i, ax in enumerate(axes):
    image, boxes, labels = train_dataset.samples[i]
    ax.imshow(image)
    for box, label in zip(boxes, labels):
        x, y, w, h = box
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="black", linewidth=1.5))
        ax.text(x, max(y - 4, 0), SHAPE_CLASSES[label], fontsize=8)
    ax.set_axis_off()
plt.suptitle("Synthetic fine-tuning examples (ground truth)")
plt.tight_layout()
plt.show()

### B5. Fine-tuning: pre-/post-comparison via qualitative boxes + `torchmetrics` mAP

A **fresh** copy of the pretrained checkpoint is loaded with `num_labels=3` (circle/square/triangle) and
`ignore_mismatched_sizes=True`: this reinitializes only `class_labels_classifier` (shape mismatch: 91+1 COCO
classes vs. 3+1 shape classes), while the box-regression head and the entire backbone/encoder/decoder keep
their real pretrained weights. Before any fine-tuning, the classification head is untrained noise, so
predictions on the eval set should be close to random; `model(pixel_values=..., labels=...)` then runs
transformers' own built-in Hungarian matcher + set-prediction loss for a few real gradient steps, and mAP is
compared before vs. after.

In [ ]:
ft_model = DetrForObjectDetection.from_pretrained(
    DETR_CHECKPOINT, num_labels=len(SHAPE_CLASSES), ignore_mismatched_sizes=True
).to(device)


@torch.no_grad()
def evaluate_map(model, dataset, feature_extractor, device):
    model.eval()
    metric = MeanAveragePrecision()
    for i in range(len(dataset)):
        pixel_values, target_labels, image = dataset[i]
        outputs = model(pixel_values=pixel_values.unsqueeze(0).to(device))
        target_sizes = torch.tensor([image.size[::-1]], device=device)
        result = feature_extractor.post_process(outputs, target_sizes=target_sizes)[0]
        pred = {"boxes": result["boxes"].cpu(), "scores": result["scores"].cpu(), "labels": result["labels"].cpu()}

        # GT boxes come back from the dataset already normalized cxcywh (DETR's training-target format);
        # torchmetrics' MeanAveragePrecision expects absolute-pixel xyxy, matching the predictions above.
        W, H = image.size
        cx, cy, w, h = target_labels["boxes"].unbind(-1)
        gt_xyxy = torch.stack([(cx - 0.5 * w) * W, (cy - 0.5 * h) * H, (cx + 0.5 * w) * W, (cy + 0.5 * h) * H], dim=-1)
        target = {"boxes": gt_xyxy, "labels": target_labels["class_labels"]}
        metric.update([pred], [target])
    return metric.compute()


@torch.no_grad()
def collect_predictions(model, dataset, feature_extractor, device, n=2):
    model.eval()
    results = []
    for i in range(min(n, len(dataset))):
        pixel_values, _, image = dataset[i]
        outputs = model(pixel_values=pixel_values.unsqueeze(0).to(device))
        target_sizes = torch.tensor([image.size[::-1]], device=device)
        result = feature_extractor.post_process(outputs, target_sizes=target_sizes)[0]
        keep = result["scores"] > 0.5
        results.append({
            "image": image,
            "boxes": result["boxes"][keep].cpu(),
            "scores": result["scores"][keep].cpu(),
            "labels": result["labels"][keep].cpu(),
        })
    return results


shape_id2label = {i: name for i, name in enumerate(SHAPE_CLASSES)}

pre_ft_map = evaluate_map(ft_model, eval_dataset, finetune_feature_extractor, device)
pre_ft_preds = collect_predictions(ft_model, eval_dataset, finetune_feature_extractor, device)
print(f"BEFORE fine-tuning: mAP = {pre_ft_map['map'].item():.4f} (mAP@50 = {pre_ft_map['map_50'].item():.4f})")

In [ ]:
opt = torch.optim.AdamW(ft_model.parameters(), lr=FINETUNE_LR)
ft_model.train()
t0 = time.time()
step = 0
for epoch in range(EPOCHS_FINETUNE):
    epoch_loss = 0.0
    for pixel_values, target_labels, _images in train_loader:
        pixel_values = pixel_values.to(device)
        target_labels = [{k: (v.to(device) if torch.is_tensor(v) else v) for k, v in t.items()}
                          for t in target_labels]
        opt.zero_grad()
        outputs = ft_model(pixel_values=pixel_values, labels=target_labels)
        outputs.loss.backward()
        opt.step()
        epoch_loss += outputs.loss.item()
        step += 1
    print(f"epoch {epoch + 1}/{EPOCHS_FINETUNE}: mean loss = {epoch_loss / len(train_loader):.4f}")
print(f"fine-tuned for {step} step(s) in {time.time() - t0:.1f}s")

In [ ]:
post_ft_map = evaluate_map(ft_model, eval_dataset, finetune_feature_extractor, device)
post_ft_preds = collect_predictions(ft_model, eval_dataset, finetune_feature_extractor, device)
print(f"BEFORE fine-tuning: mAP = {pre_ft_map['map'].item():.4f}  (mAP@50 = {pre_ft_map['map_50'].item():.4f})")
print(f"AFTER  fine-tuning: mAP = {post_ft_map['map'].item():.4f}  (mAP@50 = {post_ft_map['map_50'].item():.4f})")

# Qualitative comparison: same eval images, before vs. after.
n_show = len(pre_ft_preds)
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
axes = axes.reshape(2, n_show)
for col in range(n_show):
    plot_detections(pre_ft_preds[col]["image"], pre_ft_preds[col], shape_id2label, ax=axes[0, col],
                     title=f"eval[{col}] -- BEFORE fine-tuning")
    plot_detections(post_ft_preds[col]["image"], post_ft_preds[col], shape_id2label, ax=axes[1, col],
                     title=f"eval[{col}] -- AFTER fine-tuning")
plt.tight_layout()
plt.show()

## Summary: what set prediction + Hungarian matching buys over anchor/NMS pipelines

- **No hand-designed anchors, no NMS**: classical detectors predict many overlapping candidate boxes per
  location (anchors at multiple scales/aspect ratios, or region proposals) and rely on a *non-differentiable*
  post-processing heuristic (NMS) to collapse duplicates. DETR instead predicts a fixed, small set of boxes
  directly, one per object query, and trains that direct mapping with a *loss* -- Hungarian bipartite
  matching -- that itself enforces one-to-one query-to-object assignment during training. The direct
  by-hand-verified pipeline in Part A (cost matrix -> `linear_sum_assignment` -> matched-pair loss) *is* that
  training signal; there is no separate suppression step at inference time.
- **Why GIoU, not plain IoU, drives the matching cost**: A2/A3 showed IoU is exactly 0 -- with zero gradient
  signal -- for any pair of non-overlapping boxes, so a matcher/optimizer using IoU alone can't tell "close
  but not touching" apart from "on the opposite side of the image." GIoU stays informative (increasingly
  negative) as boxes get farther apart, which is what lets gradient-based training actually pull a
  poorly-placed prediction toward its assigned target instead of stalling with a flat 0 signal.
- **Permutation invariance (A7) is the formal justification for one-to-one matching**: since the optimal
  assignment (and its cost) doesn't depend on the arbitrary order ground-truth objects are listed in, "does
  query 3 correspond to *the second* GT box" is a meaningless question outside of a specific matching run --
  what's invariant, and what training actually optimizes, is the total matched cost and the semantic
  (query, physical object) pairing it induces.
- **The real pretrained model in Part B runs the identical mechanism**: transformers' `DetrHungarianMatcher`
  and `DetrLoss` (invoked via `model(pixel_values=..., labels=...)` in B5) implement the same cost
  construction, `linear_sum_assignment` solve, and down-weighted-no-object set-prediction loss built from
  scratch in Part A -- just at 100 queries and real image features instead of 5 toy queries. The
  cross-attention maps in B3 are a direct, visual answer to "how does a single query end up responsible for
  one specific object": each query's decoder cross-attention concentrates on the image region containing the
  object it was matched to during training.
- **Known limitations** (per the module reading list): DETR trains slowly to converge relative to anchor-based
  detectors, its (non-hierarchical) global attention over a full-resolution feature map is expensive and
  weak on small objects, and quadratic attention cost over large feature maps limits resolution -- motivating
  Deformable DETR's sparse, learned sampling-point attention as a follow-up.